In [44]:
import os
import sqlite3
from pathlib import Path
# Web scraping
from urllib.parse import urlparse, urljoin, urldefrag
import tempfile
from bs4 import BeautifulSoup as BS
from selenium.webdriver.chrome.service import Service # connecter til scraper
from webdriver_manager.chrome import ChromeDriverManager # connecter til scraper
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

basedir = os.path.abspath(os.path.dirname("__file__"))


class Config(object):
    DEBUG = False
    TESTING = False
    CSRF_ENABLED = True
    SECRET_KEY = "this-really-needs-to-be-changed"
    DATABASE_PATH = os.path.join(basedir, "jobs.db")

    def get_db(self):
        return sqlite3.connect(self.DATABASE_PATH)


class ProductionConfig(Config):
    DEBUG = False
    # Production should use a real DB via DATABASE_PATH env var or override


class StagingConfig(Config):
    DEVELOPMENT = True
    DEBUG = True


class DevelopmentConfig(Config):
    DEVELOPMENT = True
    DEBUG = True
    DATABASE_PATH = os.path.join(basedir, "dev.db")


class TestingConfig(Config):
    TESTING = True
    DATABASE_PATH = ":memory:"

In [ ]:
# import requests
# from bs4 import BeautifulSoup
# from scrape_unpaid import scrape_unpaid
# from scrape_paid import scrape_paid
# import sqlite3
# import os

db_path = os.path.join(os.path.abspath(os.path.dirname("__file__")), "jobs.db")

page_number_limit = 5000
last_known_result = None


def build_url(min_date, max_date, page_number, arkiv=0):
    return (
        "https://www.jobindex.dk/jobsoegning?"
        + "maxdate="
        + max_date
        + "&mindate="
        + min_date
        + "&page="
        + str(page_number)
        + f"&archive={arkiv}" # Inkluderer arkiverede jobopslag
    )


def scrape(max_date, min_date, page_number):
    last_known_date = max_date

    url = build_url(min_date, max_date, page_number)
    print("fetching data from url: {0}".format(url))
    page = requests.get(url)
    soup = BeautifulSoup(page.content, "html.parser")

    if soup.find("div", class_="PaidJob"): # Selve jobstillingen på jobindex
        last_known_date = (
            soup.find("div", class_="PaidJob").find("time").attrs["datetime"]
        )

    company_div = soup.find("div", class_="jix-toolbar-top__company")
    if company_div:
        company_link = company_div.find("a", target="_blank")
        if company_link:
            company_name = company_link.get_text()
            company_url = company_link.get("href")

    area_div = soup.find("div", class_="jobad-element-area")
    if area_div:
        area_span = area_div.find("span", class_="jix_robotjob--area")
        if area_span:
            location = area_span.get_text()
        p_tags = area_div.find_all("p")
        if p_tags:
            p1_text = p_tags[0].get_text()
            p2_text = p_tags[1].get_text()

    ## do page request
    if len(page.content) > 0:
        conn = sqlite3.connect(db_path)
        try:
            ## fetch unpaid
            scrape_unpaid(soup, conn)

            ## fetch paid
            scrape_paid(soup, conn)

            conn.commit()
        finally:
            conn.close()

        return last_known_date

    return False


def format_date(date):
    return ("").join((date).split("-"))


def url_format_date(date):
    return ("").join(date.split("-"))


def run(page_number, min_date, max_date):
    # run batch of scrapes
    while page_number <= page_number_limit:

        last_known_result = scrape(
            page_number=page_number, max_date=max_date, min_date=min_date
        )

        # if the requests returns content
        if last_known_result != False:
            print("\n\n\n")
            print("last known date: {0}".format(last_known_result))
            page_number += 1
        else:
            print("shows over!")
            return False

    proper_date = url_format_date(last_known_result)
    print("about to start a new batch. last known date is: {0}".format(proper_date))
    run(page_number=1, min_date=min_date, max_date=proper_date)

In [45]:
url = "https://www.jobindex.dk/jobsoegning?maxdate=20260306&mindate=20260301&page=1"

CHROMEDRIVER_PATH = ChromeDriverManager().install()

def fetch_single(url: str) -> Path | None:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.page_load_strategy = "normal"

    driver = None
    try:
        # Brug allerede-resolved driver path (ingen install her)
        service = Service(CHROMEDRIVER_PATH)
        driver = webdriver.Chrome(service=service, options=chrome_opts)

        driver.set_page_load_timeout(30)
        driver.get(url)

        # Vent på et element, der faktisk hører til jobresultaterne
        wait = WebDriverWait(driver, 20)
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "body"))
        )

        # Ekstra buffer til async rendering
        driver.implicitly_wait(2)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        return soup

    finally:
        if driver:
            driver.quit()

In [46]:
html = fetch_single(url)

In [48]:
html

<html lang="da-DK"><head>
<title>Ledige job | Jobindex</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta charset="utf-8"/>
<link href="/favicon.ico?v=2" rel="icon" sizes="any"/>
<link href="/icon.svg?v=2" rel="icon" type="image/svg+xml"/>
<link href="/apple-touch-icon.png?v=2" rel="apple-touch-icon"/>
<link href="/manifest.webmanifest" rel="manifest"/>
<link href="/res/select2/dist/css/select2.min.css?h=76de04d92330d15b62540db26b0012e3bd7204c9" rel="stylesheet"/><link href="/res/bootstrap-datepicker/dist/css/bootstrap-datepicker3.standalone.min.css?h=ab8aa700f5958fdf864d12d2aebcabdbc37d4035" rel="stylesheet"/><link href="/res/maplibre-gl/dist/maplibre-gl.css?h=aa4f4650fdf926d084d170399123c2de9244bf53" rel="stylesheet"/><link href="/css/_scss/fonts/roboto.5ea080079207ad7dd581.css" rel="stylesheet"/><link href="/css/_scss/fonts/frank_ruhl_libre.15701488e7249b2dbfe5.css" rel="stylesheet"/><link href="/css/_scss/themes/jobindex.d87ae50375c537870576.css" rel="stylesheet"/

In [ ]:
print(driver.current_url)
print(driver.title)
print(driver.execute_script("return document.readyState"))

<!DOCTYPE html>

<html lang="da-DK">
<head>
<title>Ledige job | Jobindex</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta charset="utf-8"/>
<link href="/favicon.ico?v=2" rel="icon" sizes="any"/>
<link href="/icon.svg?v=2" rel="icon" type="image/svg+xml"/>
<link href="/apple-touch-icon.png?v=2" rel="apple-touch-icon"/>
<link href="/manifest.webmanifest" rel="manifest"/>
<link href="/res/select2/dist/css/select2.min.css?h=76de04d92330d15b62540db26b0012e3bd7204c9" rel="stylesheet"/><link href="/res/bootstrap-datepicker/dist/css/bootstrap-datepicker3.standalone.min.css?h=ab8aa700f5958fdf864d12d2aebcabdbc37d4035" rel="stylesheet"/><link href="/res/maplibre-gl/dist/maplibre-gl.css?h=aa4f4650fdf926d084d170399123c2de9244bf53" rel="stylesheet"/><link href="/css/_scss/fonts/roboto.5ea080079207ad7dd581.css" rel="stylesheet"/><link href="/css/_scss/fonts/frank_ruhl_libre.15701488e7249b2dbfe5.css" rel="stylesheet"/><link href="/css/_scss/themes/jobindex.d87ae50375c537870576.css"

In [55]:
import os
os.getcwd()

'c:\\Users\\irisg\\Documents\\GitHub\\jobindex-scraper'

In [56]:
from __future__ import annotations

from pathlib import Path
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
import time


URL = "https://www.jobindex.dk/jobsoegning?maxdate=20260306&mindate=20260301&page=1"
OUT_DIR = Path("debug_jobindex")
OUT_DIR.mkdir(exist_ok=True)

CHROMEDRIVER_PATH = ChromeDriverManager().install()


def debug_fetch_jobindex(url: str, out_dir: Path) -> BeautifulSoup:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1600,2200")
    chrome_opts.page_load_strategy = "normal"

    driver = None
    try:
        service = Service(CHROMEDRIVER_PATH)
        driver = webdriver.Chrome(service=service, options=chrome_opts)

        driver.set_page_load_timeout(30)
        driver.get(url)

        wait = WebDriverWait(driver, 20)

        # Vent på body som minimum
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

        # Vent til document.readyState er complete
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

        # Ekstra buffer til async rendering
        time.sleep(3)

        # Debug-info
        current_url = driver.current_url
        title = driver.title
        ready_state = driver.execute_script("return document.readyState")
        body_text_preview = driver.find_element(By.TAG_NAME, "body").text[:1500]

        print("\n=== basic page info ===")
        print("requested url:", url)
        print("current_url  :", current_url)
        print("title        :", title)
        print("readyState   :", ready_state)

        print("\n=== body preview ===")
        print(body_text_preview)

        # Forsøg at finde elementer, der kunne være jobresultater
        candidate_selectors = [
            "[data-testid*='job']",
            "[class*='job']",
            "article",
            ".jobsearch-result",
            ".PaidJob",
            ".jix_jobad",
            ".list-item",
            "main",
        ]

        print("\n=== selector probe ===")
        for selector in candidate_selectors:
            try:
                elems = driver.find_elements(By.CSS_SELECTOR, selector)
                print(f"{selector:<30} -> {len(elems)}")
            except Exception as e:
                print(f"{selector:<30} -> ERROR: {e}")

        html = driver.page_source

        html_path = out_dir / "page.html"
        png_path = out_dir / "page.png"
        txt_path = out_dir / "body_preview.txt"

        html_path.write_text(html, encoding="utf-8")
        txt_path.write_text(body_text_preview, encoding="utf-8")
        driver.save_screenshot(str(png_path))

        print("\n=== files written ===")
        print(html_path.resolve())
        print(png_path.resolve())
        print(txt_path.resolve())

        soup = BeautifulSoup(html, "html.parser")
        return soup

    finally:
        if driver:
            driver.quit()


if __name__ == "__main__":
    soup = debug_fetch_jobindex(URL, OUT_DIR)

    print("\n=== soup checks ===")
    print("title tag:", soup.title.text.strip() if soup.title else None)

    canonical = soup.find("link", rel="canonical")
    print("canonical:", canonical.get("href") if canonical else None)

    # Kig efter ord som typisk findes på den renderede side
    text = soup.get_text(" ", strip=True)
    for needle in ["job", "Ledige job", "Din adresse", "Ingen job", "Søgning"]:
        print(f"contains {needle!r}: {needle in text}")


=== basic page info ===
requested url: https://www.jobindex.dk/jobsoegning?maxdate=20260306&mindate=20260301&page=1
current_url  : https://www.jobindex.dk/jobsoegning?maxdate=20260306&mindate=20260301
title        : Ledige job | Jobindex
readyState   : complete

=== body preview ===
34.400 job i dag
For jobsøgere
For arbejdsgivere
DA
EN
Jobsøgning
Arbejdspladser
Test dig selv
Guides
Kurser
Log ind
Opret profil
Geografi
Kategorier
Filtre
6.389 jobannoncer
TIP
Vil du modtage denne slags jobannoncer på mail?
Opret Jobagent
Kortvisning
Sorter efter:
Relevans
Skyttebjerg - det lille børnehus
Daglig leder til Skyttebjerg Børnehus
Nærum
Se rejsetid
Her har du muligheden for at stå i spidsen for dit eget børnehus, et sted, hvor du sætter den pædagogiske retning, skaber tydelige og trygge rammer og er tæt på både børn, voksne og hverdagslivet.
Samarbejdet med forældrene er kendetegnet ved åbenhed og nærvær; døren er åben og der skabes rum for fælles hverdagsliv, blandt andet gennem tilbagevend